# Capstone ? Refresh / Content Opportunity Scoring

## Abstract

This capstone builds a public-safe, decision-support queue for selecting which existing content pages a reviewer should inspect first. It measures whether observable prior-window search signals rank a defined short-window decline proxy better than a transparent rule, using client-held-out validation. Results are computed live below; they do not establish that a refresh causes recovery.

## 1. Question

Given a client?s existing content inventory, which pages should a reviewer inspect first when review capacity is limited? One row is one pseudonymized page. The output is a ranked review queue with a suggested human action and a reason code.

In [1]:
import json
from pathlib import Path
import pandas as pd
out=Path("work/outputs")
def load_json(name):
 p=out/name
 return json.loads(p.read_text()) if p.exists() else None
w05,w06,w07=load_json("w05_model_metrics.json"),load_json("w06_validation_audit.json"),load_json("w07_action_playbook_metrics.json")
print("Artifacts found:",{"w05":w05 is not None,"w06":w06 is not None,"w07":w07 is not None})


Artifacts found: {'w05': False, 'w06': False, 'w07': False}


## 2. Data

I use the verified March 2026 partition of `fact_content_daily_performance`, at daily ? client ? content grain, aggregated to one row per client/content page. March 1?15 supplies features; March 16?31 supplies only the decline-proxy label. The frame keeps pages with at least 100 prior-half impressions. Pseudonymous IDs group validation and identify approved queue rows; they are never features.

## 3. Methodology

The label is `impressions_outcome < 0.8 ? impressions_prev`. Features are prior-half impressions/clicks (log transformed), average position, position volatility, and active days. The baseline ranks prior impressions and doubles visible pages with high prior position volatility. A logistic regression is a readable reference and a random forest is the non-linear comparison. A grouped client holdout and out-of-fold client scores address repeated-client leakage.

## 4. Results (vs baseline)

The live table below compares identical held-out pages and labels. Precision@20/50 match the limited-review decision; average precision provides whole-ranking context. These are development-slice measurements, not fixed claims.

In [2]:
if w05:
 display(pd.DataFrame(w05["results"]))
else: print("Run w05_model.ipynb first to generate the model comparison receipt.")
if w06: print("Random-row vs grouped Precision@50:",w06["random_row_precision_at_50"],w06["grouped_precision_at_50"])


Run w05_model.ipynb first to generate the model comparison receipt.


## 5. Limitations

This is one monthly development partition and a short-window proxy, not an intervention study. The model does not observe page text, business value, editorial effort, or later outcomes beyond the proxy window. It cannot predict Google?s algorithm, prove a refresh will work, or replace human review. Warehouse history is an unbalanced panel, so future versions should repeat the design across periods and audit coverage.

## 6. Ranked recommendations

The action playbook ranks pseudonymized pages using out-of-fold client scores. `visible_position_unstable` calls out substantial prior visibility plus volatility; `visible_search_history` calls out meaningful prior visibility. The recommended action is review, not automatic editing.

In [3]:
queue_path=out/"w07_action_playbook_queue.csv"
if queue_path.exists():
 queue=pd.read_csv(queue_path); display(queue.head(20)); print("Queue rows:",len(queue))
else: print("Run w07_action_playbook.ipynb first to generate the review queue.")


Run w07_action_playbook.ipynb first to generate the review queue.


## 7. Artifacts the paper embeds

- Model comparison table: `work/outputs/w05_model_metrics.json`
- Validation receipt: `work/outputs/w06_validation_audit.json`
- Ranked action queue and aggregate receipt: `work/outputs/w07_action_playbook_queue.csv` and `w07_action_playbook_metrics.json`

The paper should show aggregates or approved pseudonymized examples only.

## 8. Acknowledgments & data credit

Built with the FlyRank ML Internship starter materials and the approved pseudonymized FlyRank internship warehouse release. FlyRank: https://flyrank.ai.

## 9. ML-12 closing materials

**Five-minute demo:** problem and review constraint ? data/timeline ? baseline versus model table ? validation/leakage audit ? playbook, limitations, and next test.

**Social cut:** I built a client-held-out content-review ranking workflow that compares a transparent baseline with learned scores, then turns the ranking into human-readable reason codes. The key discipline was separating the feature window from the later decline proxy and treating the result as decision-support, not an automation claim.

**Employer summary:** I framed a content-prioritization decision as a ranking task. I built a reproducible group-held-out validation and leakage audit around it. I translated the measured ranking into an explainable human-review queue with explicit limits.

## Self-check

- [x] Question, data, method, results, limits, recommendations, artifacts, and closing material are included
- [x] All metrics/rows are loaded from live notebook artifacts
- [x] Claims are observed, measured, and decision-support only
- [ ] Deploy the paper and place its direct URL in `submission/paper_url.txt`